## 1. Chuẩn bị môi trường

Bật GPU và Internet trên Kaggle. Chạy phần clone một lần khi bắt đầu phiên làm việc.

In [ ]:
!git clone https://github.com/tranminhduc9/well-log-imputation.git /kaggle/working/well-log-imputation
%cd /kaggle/working/well-log-imputation

In [ ]:
%env CUBLAS_WORKSPACE_CONFIG=:4096:8
%pip install -q -r requirements.txt

## 2. Đường dẫn và cấu hình

Lượt chạy xuất một tệp `test_model_comparison.csv` trong `/kaggle/working/models/`.
Các dòng được nhóm theo kịch bản trên tập test. Mỗi ô RMSE, MAE, MAPE, MSE và R² có dạng `trung bình ± độ lệch chuẩn mẫu`, với 4 chữ số thập phân. Cột cuối ghi số tham số.

RMSE, MAE, MSE và R² dùng thang chuẩn hóa. MAPE lấy trung bình đều của GR, RHOB và DTC ở đơn vị gốc cho từng seed. LOCF chạy một lần, có 0 tham số và mỗi ô chỉ ghi giá trị của lượt chạy đó. Số tham số của XGBoost để trống.

In [ ]:
from argparse import Namespace
import logging
from pathlib import Path
import sys

import pandas as pd
from IPython.display import FileLink, display

In [ ]:
PROJECT_ROOT = Path('/kaggle/working/well-log-imputation')
DATA_ROOT = Path('/kaggle/input/datasets/minhduc0912/geolink-dataset')
OUTPUT_ROOT = Path('/kaggle/working/models')
DEVICE = 'cuda'
MODELS = ('locf', 'xgboost', 'brits', 'saits', 'conv_saits', 'flexlognet')

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT))
from src.experiments.runner import run_experiment

In [ ]:
SEEDS = (912, 219, 129)
NEURAL_EPOCHS = 500
NEURAL_PATIENCE = 45
MODEL_SETTINGS = {
    "xgboost": {"max_depth": 3, "learning_rate": 0.03, "n_estimators": 300},
    "brits": {
        "hidden_size": 64, "learning_rate": 0.0005, "batch_size": 32,
        "validation_batch_size": 256,
        "consistency_weight": 0.1, "weight_decay": 1e-5,
        "mit_weight": 1.0, "mit_reduction": "segment",
        "masking_strategy": "mixed", "gradient_clip": 1.0,
        "validation_metric": "rmse", "min_delta": 0.0,
    },
    "saits": {
        "d_model": 256, "d_inner": 128, "n_layers": 2, "n_heads": 4,
        "learning_rate": 0.0005, "batch_size": 32,
        "ort_weight": 1.0, "mit_weight": 1.0, "mit_reduction": "segment",
        "min_delta": 0.0,
    },
    "flexlognet": {
        "encoder_channels": (32, 64, 128, 256), "resample_factors": (4, 8, 16),
        "learning_rate": 0.001, "batch_size": 32, "dropout": 0.1,
        "ssim_weight": 1.0, "statistics_weight": 0.5, "mean_weight": 4.0,
        "gradient_clip": 1.0, "min_delta": 0.0,
    },
    "conv_saits": {
        "d_model": 256, "d_inner": 128, "n_layers": 2, "n_heads": 4,
        "learning_rate": 0.0005, "batch_size": 32,
        "encoder_channels": 8, "kernel_size": 15, "conv_expansion": 2,
        "multi_scale_conv": True, "multi_scale_kernels": (3, 7, 15),
        "depth_encoding": False,
        "masking_strategy": "mixed", "mit_reduction": "segment",
        "ort_weight": 1.0, "mit_weight": 1.0, "min_delta": 0.0,
        "masking_rate": 0.2, "independent_shuffle": True,
    },
}

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s', force=True)

## 3. Huấn luyện và đánh giá

In [ ]:
args = Namespace(
    data_root=str(DATA_ROOT), output_dir=str(OUTPUT_ROOT), device=DEVICE,
    models=MODELS, seeds=SEEDS, evaluate_test=True,
    xgb_estimators=MODEL_SETTINGS['xgboost']['n_estimators'],
    brits_epochs=NEURAL_EPOCHS, brits_patience=NEURAL_PATIENCE,
    saits_epochs=NEURAL_EPOCHS, saits_patience=NEURAL_PATIENCE,
    flexlognet_epochs=NEURAL_EPOCHS, flexlognet_patience=NEURAL_PATIENCE,
    conv_saits_epochs=NEURAL_EPOCHS, conv_saits_patience=NEURAL_PATIENCE,
)

csv_path = run_experiment(args, PROJECT_ROOT, MODEL_SETTINGS, summary_only=True)
display(pd.read_csv(csv_path, keep_default_na=False))
display(FileLink(str(csv_path)))